# Phase 8 — Forecast: next quarter's active subscribers and revenue

| Series | Grain | Models | Holdout |
|---|---|---|---|
| Active subscribers | monthly (month-end) | naive, drift, Holt (damped trend) | last 3 months |
| Revenue | **daily**, summed to the quarter | seasonal naive (weekly), ETS with weekly seasonality | last 90 days |

**Why two grains?** After dropping the warm-up months there are fewer than 24 monthly points — not enough to learn a *yearly* pattern (you need at least two full cycles). Daily revenue has hundreds of points and a clear weekly/billing pattern, so it is where a seasonal model is justified. Saying *"I didn't fit yearly seasonality because two years of data can't support it"* is a strong interview answer.

Rule: always beat a simple baseline. If a fancy model can't beat *naive*, you report naive.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from streampulse.db import query, t, save_result, banner, RESULTS_DIR, IS_FIXTURE

banner()
FIG = RESULTS_DIR / "figures"
FIG.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})
import warnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
warnings.filterwarnings("ignore")

H_MONTHS, H_DAYS = 3, 90


def mae(a, f): return float(np.mean(np.abs(np.asarray(a) - np.asarray(f))))
def mape(a, f): return float(np.mean(np.abs((np.asarray(a) - np.asarray(f)) / np.asarray(a))) * 100)

## 8.1 Monthly active subscribers

In [ ]:
kpi = query(f"select * from {t('mart_monthly_kpis')} order by month_start")
kpi["month_start"] = pd.to_datetime(kpi["month_start"])
s = kpi[~kpi.is_warmup_month].set_index("month_start")["active_subscribers"].astype(float).asfreq("MS")
train, test = s.iloc[:-H_MONTHS], s.iloc[-H_MONTHS:]
print(f"train {train.index.min().date()}..{train.index.max().date()} ({len(train)} months), test {len(test)} months")

fc = {}
fc["naive"] = np.repeat(train.iloc[-1], H_MONTHS)
slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
fc["drift"] = train.iloc[-1] + slope * np.arange(1, H_MONTHS + 1)
holt = ETSModel(train, error="add", trend="add", damped_trend=True).fit(disp=False)
fc["holt_damped"] = holt.forecast(H_MONTHS).values
if len(train) >= 24:
    fc["holt_winters_yearly"] = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12).fit().forecast(H_MONTHS).values
else:
    print(f"Yearly-seasonal model skipped: {len(train)} training months < 24 (two seasonal cycles).")

subs_scores = pd.DataFrame({m: {"MAE": mae(test, f), "MAPE_%": mape(test, f)} for m, f in fc.items()}).T.sort_values("MAPE_%")
best_subs = subs_scores.index[0]
subs_scores

In [ ]:
# Refit the winning model on ALL months and forecast the next quarter with an interval
full = s
if best_subs == "holt_damped":
    final = ETSModel(full, error="add", trend="add", damped_trend=True).fit(disp=False)
    pred = final.get_prediction(start=len(full), end=len(full) + H_MONTHS - 1).summary_frame(alpha=0.05)
    nxt = pd.DataFrame({"forecast": pred["mean"].values, "low_95": pred["pi_lower"].values, "high_95": pred["pi_upper"].values})
else:
    # naive / drift: interval from the spread of one-step changes (simple, stated)
    steps = np.arange(1, H_MONTHS + 1)
    sd = full.diff().dropna().std()
    base = full.iloc[-1] + (slope * steps if best_subs == "drift" else 0)
    nxt = pd.DataFrame({"forecast": base, "low_95": base - 1.96 * sd * np.sqrt(steps), "high_95": base + 1.96 * sd * np.sqrt(steps)})
nxt.index = pd.date_range(full.index[-1] + pd.offsets.MonthBegin(1), periods=H_MONTHS, freq="MS")

fig, ax = plt.subplots()
ax.plot(s.index, s, label="actual", color="black")
for m, f in fc.items():
    ax.plot(test.index, f, ls="--", label=f"{m} (holdout)")
ax.plot(nxt.index, nxt.forecast, color="#08519c", lw=2, label=f"next quarter: {best_subs}")
ax.fill_between(nxt.index, nxt.low_95, nxt.high_95, color="#08519c", alpha=0.15)
ax.set_title("Active subscribers (month-end)"); ax.legend(fontsize=8)
fig.savefig(FIG / "p08_subscribers_forecast.png", dpi=150, bbox_inches="tight")
nxt.round(0)

## 8.2 Daily revenue with weekly seasonality → next-quarter revenue

In [ ]:
dr = query(f"select * from {t('mart_daily_revenue')} order by revenue_date")
dr["revenue_date"] = pd.to_datetime(dr["revenue_date"])
start = kpi.loc[~kpi.is_warmup_month, "month_start"].min()
d = dr.set_index("revenue_date")["revenue_ntd"].astype(float).asfreq("D", fill_value=0.0)
d = d[d.index >= start]
dtrain, dtest = d.iloc[:-H_DAYS], d.iloc[-H_DAYS:]

dfc = {}
dfc["seasonal_naive_weekly"] = np.tile(dtrain.iloc[-7:].values, H_DAYS // 7 + 1)[:H_DAYS]
dfc["ets_weekly"] = ExponentialSmoothing(dtrain, trend="add", damped_trend=True, seasonal="add",
                                         seasonal_periods=7).fit().forecast(H_DAYS).values
rev_scores = pd.DataFrame({m: {"daily_MAE": mae(dtest, f),
                               "quarter_total_actual": float(dtest.sum()), "quarter_total_forecast": float(np.sum(f)),
                               "quarter_total_error_%": float((np.sum(f) - dtest.sum()) / dtest.sum() * 100)}
                           for m, f in dfc.items()}).T
rev_scores["abs_quarter_error_%"] = rev_scores["quarter_total_error_%"].abs()
best_rev = rev_scores.sort_values("abs_quarter_error_%").index[0]
rev_scores

In [ ]:
if best_rev == "ets_weekly":
    dfinal = ExponentialSmoothing(d, trend="add", damped_trend=True, seasonal="add", seasonal_periods=7).fit()
    dnext = dfinal.forecast(H_DAYS)
else:
    dnext = pd.Series(np.tile(d.iloc[-7:].values, H_DAYS // 7 + 1)[:H_DAYS],
                      index=pd.date_range(d.index[-1] + pd.Timedelta(days=1), periods=H_DAYS))
holdout_err = rev_scores.loc[best_rev, "abs_quarter_error_%"] / 100
q_total = float(dnext.sum())
print(f"Next-quarter revenue ({best_rev}): {q_total:,.0f} NTD; +/- {holdout_err:.1%} based on the holdout quarter's error")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(d.index[-180:], d.iloc[-180:], color="black", lw=0.8, label="actual")
ax.plot(dtest.index, dfc[best_rev], color="#de2d26", lw=0.8, label=f"{best_rev} (holdout)")
ax.plot(dnext.index, dnext.values, color="#08519c", lw=0.8, label="next quarter")
ax.set_title("Daily revenue (NTD)"); ax.legend(fontsize=8)
fig.savefig(FIG / "p08_revenue_forecast.png", dpi=150, bbox_inches="tight")

In [ ]:
res = {
    "subscribers": {"train_months": [str(train.index.min().date()), str(train.index.max().date())],
                    "holdout_months": H_MONTHS, "scores": subs_scores.round(4).to_dict("index"),
                    "best_model": best_subs,
                    "next_quarter": {str(k.date()): {c: float(v) for c, v in r.items()} for k, r in nxt.iterrows()}},
    "revenue": {"holdout_days": H_DAYS, "scores": rev_scores.round(4).to_dict("index"), "best_model": best_rev,
                "next_quarter_total_ntd": q_total, "holdout_abs_error_pct": float(holdout_err * 100),
                "next_quarter_dates": [str(dnext.index.min().date()), str(dnext.index.max().date())]},
    "yearly_seasonality_fitted": "holt_winters_yearly" in fc,
}
save_result("phase08_forecast", res)
res